# 1.4 Recurrent variant frequency across samples

This notebook checks how often the same exact variant appears across samples within one sequencing run.

To switch runs, change the `run` variable in the setup cell.

A variant is considered the same if these columns match:

- `Chr`
- `Start`
- `REF`
- `ALT`

Because the merged file contains both DeepVariant and Mutect2 calls, duplicate calls of the same variant in the same sample are removed before counting frequency.

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

## Setup

Change `run` to `"Run2"`, `"Run3"`, or `"Run4"` depending on which run you want to analyze.

In [ ]:
runs = ["Run2"]
#["Run1", "Run2", "Run3", "Run4"]

input_dir = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/04_qc_checking_on_target")

base_output_dir = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/01.4_Variant_frequency_per_sample/NewGnomadAnnotations")
base_output_dir.mkdir(parents=True, exist_ok=True)

variant_cols = ["Chr", "Start", "REF", "ALT"]

TARGET_GENES = ["ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1", "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53"]
zoom_max=50

## Count variant frequency across samples for each run

For each run, this cell:

1. Loads the merged per-variant table.
2. Keeps only `PASS` variants if the `FILTER` column is present.
3. Counts the number of unique samples in the run.
4. Removes duplicate calls of the same variant within the same sample.
5. Counts how many samples contain each exact variant.
6. Calculates the percent of samples in the run with each variant.
7. Saves one full variant frequency table and one table containing variants found in fewer than 50 samples.

Duplicate caller calls are removed because the merged file contains both DeepVariant and Mutect2. If both callers found the same variant in the same sample, that sample should only count once.

In [ ]:
def add_variant_frequency_columns(input_df, run):
    n_samples = input_df["Sample.ID"].nunique()

    one_row_per_sample_variant = input_df.drop_duplicates(
        subset=["Sample.ID"] + variant_cols
    )

    frequency_cols = (
        one_row_per_sample_variant
        .groupby(variant_cols)
        .agg(sample_count=("Sample.ID", "nunique"))
        .reset_index()
    )

    frequency_cols["total_samples_in_run"] = n_samples
    frequency_cols["sample_fraction"] = (
        frequency_cols["sample_count"] / n_samples
    )
    frequency_cols["sample_percent"] = (
        frequency_cols["sample_fraction"] * 100
    ).round(2)

    output_df = input_df.merge(
        frequency_cols,
        on=variant_cols,
        how="left"
    )

    return output_df

## Plot variant frequency histogram

This plot shows how many unique variants appear in 1 sample, 2 samples, 3 samples, etc.

The x-axis is the number of samples containing the variant.

The y-axis is the number of unique variants with that sample frequency.

Because the dataframe still contains the original rows, we first keep only one row per unique variant before plotting.

In [ ]:
def plot_variant_frequency_histogram(input_df, title, out_png):
    unique_variants = input_df.drop_duplicates(subset=variant_cols).copy()

    n_samples = int(unique_variants["total_samples_in_run"].iloc[0])

    freq_counts = (
        unique_variants["sample_count"]
        .astype(int)
        .value_counts()
        .reindex(range(1, n_samples + 1), fill_value=0)
        .sort_index()
    )

    # Full plot
    plt.figure(figsize=(12, 5))
    plt.bar(freq_counts.index, freq_counts.values)

    plt.xlabel("Number of samples with variant")
    plt.ylabel("Number of unique variants")
    plt.title(title)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.show()

    print("Saved:", out_png)

    # Zoomed plot
    zoom_counts = freq_counts.loc[1:zoom_max]

    zoom_png = out_png.with_name(out_png.stem + f"_zoom_1_to_{zoom_max}" + out_png.suffix)

    plt.figure(figsize=(12, 5))
    plt.bar(zoom_counts.index, zoom_counts.values)

    plt.xlabel("Number of samples with variant")
    plt.ylabel("Number of unique variants")
    plt.title(title + f" (zoomed: 1–{zoom_max} samples)")

    plt.tight_layout()
    plt.savefig(zoom_png, dpi=300)
    plt.show()

    print("Saved:", zoom_png)

In [ ]:
for run in runs:
    #input_csv = input_dir / f"04_{run}_per_variant_target_status_FULL.csv"
    input_csv = Path(r"/project/knathans_shared/donetski/Notebooks/OutputFiles/06_annovar_output/fixed_annovar_output_with_gnomad41_genome_and_exome.csv")
    run_output_dir = base_output_dir / run
    csv_dir = run_output_dir / "csv"
    figure_dir = run_output_dir / "figures"
    
    csv_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)

    print("Processing:", run)
    #df = pd.read_csv(input_csv, low_memory=False)
    df = pd.read_csv(input_csv, low_memory=False, encoding="latin1")
    df = df[df["FILTER"].eq("PASS")].copy()

    #debugging lines
    df = pd.read_csv(input_csv, low_memory=False)
    
    full_input_samples = df["Sample.ID"].nunique()
    print("Full input samples:", full_input_samples)
    
    if "FILTER" in df.columns:
        df = df[df["FILTER"].eq("PASS")].copy()
    
    all_gene_samples_after_pass = df["Sample.ID"].nunique()
    
    target_df = df[df["Gene"].isin(TARGET_GENES)].copy()
    target_gene_samples_after_pass = target_df["Sample.ID"].nunique()
    
    print("All-gene PASS samples:", all_gene_samples_after_pass)
    print("16-gene PASS samples:", target_gene_samples_after_pass)
    
    assert target_gene_samples_after_pass <= all_gene_samples_after_pass

    #end debugging lines

    # All genes version: keeps all original columns + frequency columns
    all_genes_df = add_variant_frequency_columns(df, run)# Unique variant versions: one row per Chr/Start/REF/ALT
    all_genes_unique_df = all_genes_df.drop_duplicates(subset=variant_cols).copy()
    target_genes_unique_df = target_genes_df.drop_duplicates(subset=variant_cols).copy()

    print("Saved full and unique CSVs for:", run)
    # 16 target genes version: keeps all original columns + frequency columns
    target_df = df[df["Gene"].isin(TARGET_GENES)].copy()
    target_genes_df = add_variant_frequency_columns(target_df, run)

    all_genes_df.to_csv(csv_dir / f"{run}_all_genes_with_variant_frequency.csv", index=False)
    target_genes_df.to_csv(csv_dir / f"{run}_16_genes_with_variant_frequency.csv", index=False)

     # Save unique variant files
    all_genes_unique_df.to_csv(
        csv_dir / f"{run}_all_genes_unique_variant_frequency.csv",
        index=False
    )
    
    target_genes_unique_df.to_csv(
        csv_dir / f"{run}_16_genes_unique_variant_frequency.csv",
        index=False
    )
    
    plot_variant_frequency_histogram(all_genes_df, f"{run}: unique variant frequency across samples, all genes",
        figure_dir / f"{run}_all_genes_frequency_histogram.png")

    plot_variant_frequency_histogram(target_genes_df, f"{run}: unique variant frequency across samples, 16 target genes",
        figure_dir / f"{run}_16_genes_frequency_histogram.png")

    print("Saved all-gene dataframe with frequency columns")
    print("Saved 16-gene dataframe with frequency columns")

## Review run-level summary

This table summarizes the recurrent variant results for each run.
The main columns are:

- `sample_count`: number of samples in the run that contain that exact variant
- `total_samples_in_run`: total number of unique samples in that run
- `sample_fraction`: fraction of samples with that variant, calculated as `sample_count / total_samples_in_run`
- `sample_percent`: percent of samples with that variant
- `Gene`: gene annotation for the variant

In [ ]:
all_genes_unique_df.head(20)

In [ ]:
target_genes_unique_df.head(20)

In [ ]:
csv_path = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/06_annovar_output/fixed_annovar_output_with_gnomad41_genome_and_exome.csv")
df = pd.read_csv(csv_path, encoding="latin1", low_memory=False)

print("Rows:", len(df))
print("Columns:", df.shape[1])

In [ ]:
df.head()

In [ ]:
pd.set_option('display.max_columns', None)


In [ ]:
unique_sample_ids = sorted(df["Sample.ID"].dropna().unique())

print("Number of unique Sample.IDs in this file:", len(unique_sample_ids))
unique_sample_ids